In [1]:
# Step 1 — imports and project paths

from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../data")

print("Data directory:", DATA_DIR.resolve())

Data directory: /Users/sofiamartinez/caleb-williams-hamstring-rtp/data


In [2]:
# Step 2 — load finalized datasets

master = pd.read_csv(
    DATA_DIR / "qb_hamstring_episodes_final.csv"
)

stage1 = pd.read_csv(
    DATA_DIR / "stage1_missed_game_model.csv"
)

stage2 = pd.read_csv(
    DATA_DIR / "stage2_rtp_observed.csv"
)

severity = pd.read_csv(
    DATA_DIR / "severity_validation.csv"
)

print("Master:", master.shape)
print("Stage 1:", stage1.shape)
print("Stage 2:", stage2.shape)
print("Severity:", severity.shape)

Master: (29, 40)
Stage 1: (24, 40)
Stage 2: (8, 40)
Severity: (29, 11)


In [3]:
# Step 3 — verify loaded data

print("Stage 1 target:")
print(
    stage1["missed_game_target"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nStage 2 RTP target:")
print(
    stage2["rtp_games_missed"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nAvailable Stage 1 columns:")
print(stage1.columns.tolist())

Stage 1 target:
missed_game_target
0     9
1    15
Name: count, dtype: int64

Stage 2 RTP target:
rtp_games_missed
1    5
2    2
5    1
Name: count, dtype: int64

Available Stage 1 columns:
['injury_id_2wk', 'player', 'gsis_id', 'season', 'start_week', 'last_report_week', 'report_rows', 'initial_practice_status', 'initial_report_status', 'stat_weeks', 'team', 'team_game_weeks', 'injury_game_weeks', 'first_game_back', 'games_missed', 'weeks_to_return', 'missed_game_weeks', 'rtp_status', 'validated_outcome', 'validated_return_week', 'validation_reason', 'final_rtp_outcome', 'eligible_missed_game_model', 'eligible_rtp_duration_model', 'has_schedule_coverage', 'missed_game_target', 'rtp_week', 'rtp_games_missed', 'injury_grade', 'injury_severity', 'severity_source', 'severity_notes', 'reported_grade', 'reported_severity', 'severity_evidence', 'source_quality', 'source', 'notes', 'severity_group', 'injury_classification_flag']


In [ ]:
##prediction time v outcome columns

In [5]:
# Step 4 — define prediction-time and leakage columns

# Information available at or before the initial injury report
prediction_time_columns = [
    "injury_id_2wk",
    "player",
    "gsis_id",
    "season",
    "start_week",
    "team",
    "initial_practice_status",
    "initial_report_status"
]

# Information that occurs after injury onset or directly describes the outcome
leakage_columns = [
    "last_report_week",
    "report_rows",
    "stat_weeks",
    "team_game_weeks",
    "injury_game_weeks",
    "missed_game_weeks",
    "games_missed",
    "first_game_back",
    "weeks_to_return",
    "rtp_week",
    "rtp_games_missed",
    "rtp_status",
    "final_rtp_outcome",
    "validated_outcome",
    "validated_return_week"
]

print("Prediction-time columns:")
for col in prediction_time_columns:
    print("✓", col if col in stage1.columns else f"{col} — NOT FOUND")

print("\nLeakage columns found:")
for col in leakage_columns:
    if col in stage1.columns:
        print("BLOCK:", col)

Prediction-time columns:
✓ injury_id_2wk
✓ player
✓ gsis_id
✓ season
✓ start_week
✓ team
✓ initial_practice_status
✓ initial_report_status

Leakage columns found:
BLOCK: last_report_week
BLOCK: report_rows
BLOCK: stat_weeks
BLOCK: team_game_weeks
BLOCK: injury_game_weeks
BLOCK: missed_game_weeks
BLOCK: games_missed
BLOCK: first_game_back
BLOCK: weeks_to_return
BLOCK: rtp_week
BLOCK: rtp_games_missed
BLOCK: rtp_status
BLOCK: final_rtp_outcome
BLOCK: validated_outcome
BLOCK: validated_return_week


In [6]:
# Step 5 — engineer prior hamstring injury history

master_sorted = master.sort_values(
    ["player", "season", "start_week"]
).copy()

master_sorted["prior_hamstring_episodes"] = (
    master_sorted
    .groupby("player")
    .cumcount()
)

prior_history = master_sorted[
    [
        "player",
        "season",
        "start_week",
        "prior_hamstring_episodes"
    ]
].copy()

stage1 = stage1.merge(
    prior_history,
    on=["player", "season", "start_week"],
    how="left",
    validate="one_to_one"
)

print(
    stage1[
        [
            "player",
            "season",
            "start_week",
            "prior_hamstring_episodes",
            "missed_game_target"
        ]
    ]
    .sort_values(["player", "season", "start_week"])
    .to_string(index=False)
)

         player  season  start_week  prior_hamstring_episodes  missed_game_target
  Aaron Rodgers    2014          10                         0                   0
  Aaron Rodgers    2016          13                         1                   0
 Blaine Gabbert    2013           6                         0                   1
     Brad Smith    2011          17                         0                   1
  Carson Palmer    2016           7                         0                   0
   Dak Prescott    2024          10                         0                   1
   Daniel Jones    2020          13                         0                   1
  David Garrard    2010          15                         0                   0
Jacoby Brissett    2021           6                         0                   0
Jacoby Brissett    2023          17                         1                   1
     Jay Cutler    2015           3                         0                   1
   Jeff Driskel 

In [ ]:
##prior_hamstring_episode represents the # of earlier hamstring injury episodes observed for that quarterback within this project's dataset before the current episode. It does not represent the player's complete lifetime hamstring injury history.

In [7]:
# Step 6 — inspect available project data files

print("Files currently in data folder:\n")

for path in sorted(DATA_DIR.iterdir()):
    print(path.name)

Files currently in data folder:

processed
qb_hamstring_episodes_final.csv
raw
severity_validation.csv
stage1_missed_game_model.csv
stage2_rtp_observed.csv


In [8]:
# Step 7 — inspect raw and processed data folders

for folder_name in ["raw", "processed"]:
    folder = DATA_DIR / folder_name

    print(f"\n{folder_name.upper()} FILES:")
    print("-" * 50)

    if folder.exists():
        files = list(folder.rglob("*"))

        if files:
            for path in sorted(files):
                if path.is_file():
                    print(path.relative_to(DATA_DIR))
        else:
            print("(folder is empty)")
    else:
        print("(folder does not exist)")


RAW FILES:
--------------------------------------------------
(folder is empty)

PROCESSED FILES:
--------------------------------------------------
(folder is empty)


In [9]:
# Step 8 — download and save nflverse weekly player stats

RAW_DIR = DATA_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

PLAYER_STATS_PATH = RAW_DIR / "player_stats.csv"

player_stats_url = (
    "https://github.com/nflverse/nflverse-data/releases/download/"
    "player_stats/player_stats.csv"
)

print("Downloading nflverse player stats...")

player_stats = pd.read_csv(player_stats_url)

player_stats.to_csv(
    PLAYER_STATS_PATH,
    index=False
)

print("\nPlayer stats shape:", player_stats.shape)
print("Saved to:", PLAYER_STATS_PATH)

print("\nSeason range:")
print(
    player_stats["season"].min(),
    "to",
    player_stats["season"].max()
)


Player stats shape: (134470, 53)
Saved to: ../data/raw/player_stats.csv

Season range:
1999 to 2024


In [10]:
# Step 9 — validate player-stat fields

required_columns = [
    "player_id",
    "player_display_name",
    "position",
    "recent_team",
    "season",
    "week",
    "season_type",
    "attempts",
    "carries"
]

print("Required workload fields:\n")

for col in required_columns:
    if col in player_stats.columns:
        print(f"✓ {col}")
    else:
        print(f"✗ {col} — MISSING")

print("\nQB rows:")
print(
    (player_stats["position"] == "QB").sum()
)

print("\nRegular-season QB rows:")
print(
    (
        (player_stats["position"] == "QB") &
        (player_stats["season_type"] == "REG")
    ).sum()
)

Required workload fields:

✓ player_id
✓ player_display_name
✓ position
✓ recent_team
✓ season
✓ week
✓ season_type
✓ attempts
✓ carries

QB rows:
16732

Regular-season QB rows:
16063


In [11]:
# Step 10 — inspect available workload-related columns

workload_keywords = [
    "attempt",
    "carry",
    "rush",
    "pass",
    "sack",
    "snap"
]

workload_candidates = [
    col for col in player_stats.columns
    if any(keyword in col.lower() for keyword in workload_keywords)
]

print("Potential workload-related columns:\n")

for col in workload_candidates:
    print(col)

Potential workload-related columns:

attempts
passing_yards
passing_tds
sacks
sack_yards
sack_fumbles
sack_fumbles_lost
passing_air_yards
passing_yards_after_catch
passing_first_downs
passing_epa
passing_2pt_conversions
rushing_yards
rushing_tds
rushing_fumbles
rushing_fumbles_lost
rushing_first_downs
rushing_epa
rushing_2pt_conversions


In [12]:
# Step 11 — create regular-season QB workload table

qb_stats = (
    player_stats.loc[
        (player_stats["position"] == "QB") &
        (player_stats["season_type"] == "REG")
    ]
    .copy()
)

qb_stats = qb_stats.sort_values(
    ["player_id", "season", "week"]
).reset_index(drop=True)

print("QB regular-season rows:", len(qb_stats))
print("Unique QBs:", qb_stats["player_id"].nunique())

print("\nSeason coverage:")
print(
    qb_stats.groupby("season")
    .size()
    .tail(10)
)

QB regular-season rows: 16063
Unique QBs: 397

Season coverage:
season
2015    593
2016    591
2017    583
2018    612
2019    589
2020    620
2021    667
2022    648
2023    663
2024    664
dtype: int64


In [13]:
# Step 12 — check workload coverage for Stage 1 injury episodes

workload_ids = set(qb_stats["player_id"].dropna())

stage1["has_player_stats"] = (
    stage1["gsis_id"].isin(workload_ids)
)

coverage = (
    stage1["has_player_stats"]
    .value_counts(dropna=False)
)

print("Stage 1 player-stat coverage:")
print(coverage)

print("\nEpisodes without player-stat coverage:")

print(
    stage1.loc[
        ~stage1["has_player_stats"],
        [
            "player",
            "season",
            "start_week",
            "gsis_id"
        ]
    ]
    .sort_values(["season", "player"])
    .to_string(index=False)
)

Stage 1 player-stat coverage:
has_player_stats
True    24
Name: count, dtype: int64

Episodes without player-stat coverage:
Empty DataFrame
Columns: [player, season, start_week, gsis_id]
Index: []


In [ ]:
##use games before start_week, so injury week cannot leak into predictors 

In [14]:
# Step 13 — collect pre-injury regular-season QB games

def get_preinjury_games(row):
    player_id = row["gsis_id"]
    season = row["season"]
    start_week = row["start_week"]

    history = qb_stats.loc[
        (qb_stats["player_id"] == player_id) &
        (qb_stats["season"] == season) &
        (qb_stats["week"] < start_week)
    ].copy()

    return history.sort_values("week")


preinjury_summary = []

for _, row in stage1.iterrows():
    history = get_preinjury_games(row)

    preinjury_summary.append({
        "player": row["player"],
        "season": row["season"],
        "start_week": row["start_week"],
        "preinjury_games_available": len(history),
        "last_preinjury_week": (
            history["week"].max()
            if len(history) > 0
            else pd.NA
        )
    })

preinjury_summary = pd.DataFrame(preinjury_summary)

print(
    preinjury_summary
    .sort_values(
        ["preinjury_games_available", "season", "player"]
    )
    .to_string(index=False)
)

         player  season  start_week  preinjury_games_available last_preinjury_week
       Joe Webb    2010          14                          0                <NA>
     Jay Cutler    2015           3                          2                   2
Jacoby Brissett    2023          17                          2                  16
 Blaine Gabbert    2013           6                          3                   5
   Jeff Driskel    2019          13                          3                  12
Taylor Heinicke    2023          12                          3                  10
 Johnny Manziel    2014          17                          4                  16
 Marcus Mariota    2017           5                          4                   4
   Michael Vick    2013           6                          5                   5
   Michael Vick    2015           7                          5                   6
  Carson Palmer    2016           7                          5                   6
Jaco

In [15]:
# Step 14 — inspect 3-game pre-injury workload coverage

preinjury_summary["has_1_prior_game"] = (
    preinjury_summary["preinjury_games_available"] >= 1
)

preinjury_summary["has_3_prior_games"] = (
    preinjury_summary["preinjury_games_available"] >= 3
)

print("Pre-injury workload coverage:\n")

print(
    preinjury_summary[
        [
            "has_1_prior_game",
            "has_3_prior_games"
        ]
    ]
    .value_counts()
)

print("\nGames available summary:")
print(
    preinjury_summary["preinjury_games_available"]
    .describe()
)

print("\nEpisodes with fewer than 3 prior games:")

print(
    preinjury_summary.loc[
        ~preinjury_summary["has_3_prior_games"],
        [
            "player",
            "season",
            "start_week",
            "preinjury_games_available"
        ]
    ]
    .to_string(index=False)
)

Pre-injury workload coverage:

has_1_prior_game  has_3_prior_games
True              True                 21
                  False                 2
False             False                 1
Name: count, dtype: int64

Games available summary:
count    24.000000
mean      6.666667
std       3.974557
min       0.000000
25%       3.750000
50%       5.500000
75%       9.500000
max      15.000000
Name: preinjury_games_available, dtype: float64

Episodes with fewer than 3 prior games:
         player  season  start_week  preinjury_games_available
     Jay Cutler    2015           3                          2
       Joe Webb    2010          14                          0
Jacoby Brissett    2023          17                          2


In [16]:
# Step 15 — engineer pre-injury 3-game workload features

workload_features = []

for _, row in stage1.iterrows():

    history = get_preinjury_games(row)

    # Take the QB's last 3 played games before injury
    recent = history.tail(3)

    if len(recent) > 0:
        pass_attempts_avg = recent["attempts"].mean()
        carries_avg = recent["carries"].mean()

        total_actions_avg = (
            recent["attempts"] + recent["carries"]
        ).mean()

        games_used = len(recent)

    else:
        pass_attempts_avg = np.nan
        carries_avg = np.nan
        total_actions_avg = np.nan
        games_used = 0

    workload_features.append({
        "injury_id_2wk": row["injury_id_2wk"],
        "pre3_games_used": games_used,
        "pre3_pass_attempts_avg": pass_attempts_avg,
        "pre3_carries_avg": carries_avg,
        "pre3_total_actions_avg": total_actions_avg
    })

workload_features = pd.DataFrame(workload_features)

stage1_features = stage1.merge(
    workload_features,
    on="injury_id_2wk",
    how="left",
    validate="one_to_one"
)

print(
    stage1_features[
        [
            "player",
            "season",
            "start_week",
            "pre3_games_used",
            "pre3_pass_attempts_avg",
            "pre3_carries_avg",
            "pre3_total_actions_avg",
            "missed_game_target"
        ]
    ]
    .sort_values(["season", "player", "start_week"])
    .to_string(index=False)
)

         player  season  start_week  pre3_games_used  pre3_pass_attempts_avg  pre3_carries_avg  pre3_total_actions_avg  missed_game_target
    Vince Young    2009          15                3               31.333333          3.000000               34.333333                   0
  David Garrard    2010          15                3               25.333333          7.000000               32.333333                   0
       Joe Webb    2010          14                0                     NaN               NaN                     NaN                   0
     Brad Smith    2011          17                3                0.000000          0.000000                0.000000                   1
 Blaine Gabbert    2013           6                3               28.666667          3.000000               31.666667                   1
   Michael Vick    2013           6                3               23.666667          6.333333               30.000000                   1
  Aaron Rodgers    2014    

In [17]:
# Step 16 — inspect onset injury-status categories

print("Initial practice status:")
print(
    stage1_features["initial_practice_status"]
    .value_counts(dropna=False)
)

print("\nInitial report status:")
print(
    stage1_features["initial_report_status"]
    .value_counts(dropna=False)
)

print("\nCross-tab with missed-game target:")
print(
    pd.crosstab(
        [
            stage1_features["initial_practice_status"],
            stage1_features["initial_report_status"]
        ],
        stage1_features["missed_game_target"],
        margins=True
    )
)

Initial practice status:
initial_practice_status
Limited Participation in Practice    13
Did Not Participate In Practice       5
Full Participation in Practice        4
Out (Definitely Will Not Play)        1
\n                                    1
Name: count, dtype: int64

Initial report status:
initial_report_status
Questionable    16
Out              6
Probable         1
Doubtful         1
Name: count, dtype: int64

Cross-tab with missed-game target:
missed_game_target                                       0   1  All
initial_practice_status           initial_report_status            
\n                                Questionable           0   1    1
Did Not Participate In Practice   Out                    0   5    5
Full Participation in Practice    Probable               1   0    1
                                  Questionable           2   1    3
Limited Participation in Practice Doubtful               0   1    1
                                  Questionable           6   6   

In [18]:
# Step 17 — descriptive outcome rates by initial injury status

practice_summary = (
    stage1_features
    .groupby("initial_practice_status", dropna=False)
    ["missed_game_target"]
    .agg(
        episodes="count",
        missed_games="sum",
        missed_game_rate="mean"
    )
    .sort_values("episodes", ascending=False)
)

report_summary = (
    stage1_features
    .groupby("initial_report_status", dropna=False)
    ["missed_game_target"]
    .agg(
        episodes="count",
        missed_games="sum",
        missed_game_rate="mean"
    )
    .sort_values("episodes", ascending=False)
)

print("Practice-status summary:")
print(practice_summary)

print("\nReport-status summary:")
print(report_summary)

Practice-status summary:
                                   episodes  missed_games  missed_game_rate
initial_practice_status                                                    
Limited Participation in Practice        13             7          0.538462
Did Not Participate In Practice           5             5          1.000000
Full Participation in Practice            4             1          0.250000
\n                                        1             1          1.000000
Out (Definitely Will Not Play)            1             1          1.000000

Report-status summary:
                       episodes  missed_games  missed_game_rate
initial_report_status                                          
Questionable                 16             8               0.5
Out                           6             6               1.0
Doubtful                      1             1               1.0
Probable                      1             0               0.0


In [19]:
# Step 18 — validate engineered workload features

workload_cols = [
    "pre3_pass_attempts_avg",
    "pre3_carries_avg",
    "pre3_total_actions_avg"
]

print("Missing values:")
print(stage1_features[workload_cols].isna().sum())

print("\nWorkload summary:")
print(
    stage1_features[workload_cols]
    .describe()
    .round(2)
)

print("\nAverage workload by outcome:")
print(
    stage1_features
    .groupby("missed_game_target")[workload_cols]
    .mean()
    .round(2)
)

Missing values:
pre3_pass_attempts_avg    1
pre3_carries_avg          1
pre3_total_actions_avg    1
dtype: int64

Workload summary:
       pre3_pass_attempts_avg  pre3_carries_avg  pre3_total_actions_avg
count                   23.00             23.00                   23.00
mean                    27.29              4.39                   31.68
std                     10.09              2.31                   10.96
min                      0.00              0.00                    0.00
25%                     23.17              3.00                   28.17
50%                     28.67              4.00                   33.33
75%                     34.00              6.33                   38.67
max                     43.67              9.00                   47.67

Average workload by outcome:
                    pre3_pass_attempts_avg  pre3_carries_avg  \
missed_game_target                                             
0                                    33.88              4.25  

In [20]:
# Step 19 — clean initial practice status

stage1_features["initial_practice_status_clean"] = (
    stage1_features["initial_practice_status"]
    .replace({
        "\\n": pd.NA,
        "\n": pd.NA
    })
)

print(
    stage1_features["initial_practice_status_clean"]
    .value_counts(dropna=False)
)

initial_practice_status_clean
Limited Participation in Practice    13
Did Not Participate In Practice       5
Full Participation in Practice        4
Out (Definitely Will Not Play)        1
\n                                    1
Name: count, dtype: int64


In [21]:
# Step 20 — engineer compact onset injury-status features

stage1_features["practice_dnp_or_out"] = (
    stage1_features["initial_practice_status_clean"]
    .isin([
        "Did Not Participate in Practice",
        "Out (Definitely Will Not Play)"
    ])
    .astype(int)
)

stage1_features["report_out_or_doubtful"] = (
    stage1_features["initial_report_status"]
    .isin([
        "Out",
        "Doubtful"
    ])
    .astype(int)
)

print("Practice DNP/Out:")
print(
    pd.crosstab(
        stage1_features["practice_dnp_or_out"],
        stage1_features["missed_game_target"],
        margins=True
    )
)

print("\nReport Out/Doubtful:")
print(
    pd.crosstab(
        stage1_features["report_out_or_doubtful"],
        stage1_features["missed_game_target"],
        margins=True
    )
)

Practice DNP/Out:
missed_game_target   0   1  All
practice_dnp_or_out            
0                    9  14   23
1                    0   1    1
All                  9  15   24

Report Out/Doubtful:
missed_game_target      0   1  All
report_out_or_doubtful            
0                       9   8   17
1                       0   7    7
All                     9  15   24


In [22]:
# Step 21 — compare onset severity indicators

print("Practice vs. report severity indicators:")
print(
    pd.crosstab(
        stage1_features["practice_dnp_or_out"],
        stage1_features["report_out_or_doubtful"],
        margins=True
    )
)

print("\nEpisode-level comparison:")
print(
    stage1_features[
        [
            "player",
            "season",
            "start_week",
            "initial_practice_status_clean",
            "initial_report_status",
            "practice_dnp_or_out",
            "report_out_or_doubtful",
            "prior_hamstring_episodes",
            "pre3_total_actions_avg",
            "missed_game_target"
        ]
    ]
    .sort_values(["season", "player", "start_week"])
    .to_string(index=False)
)

Practice vs. report severity indicators:
report_out_or_doubtful   0  1  All
practice_dnp_or_out               
0                       17  6   23
1                        0  1    1
All                     17  7   24

Episode-level comparison:
         player  season  start_week     initial_practice_status_clean initial_report_status  practice_dnp_or_out  report_out_or_doubtful  prior_hamstring_episodes  pre3_total_actions_avg  missed_game_target
    Vince Young    2009          15    Full Participation in Practice          Questionable                    0                       0                         0               34.333333                   0
  David Garrard    2010          15 Limited Participation in Practice          Questionable                    0                       0                         0               32.333333                   0
       Joe Webb    2010          14 Limited Participation in Practice          Questionable                    0                       0

In [23]:
# Step 22 — normalize practice-status strings

stage1_features["initial_practice_status_clean"] = (
    stage1_features["initial_practice_status"]
    .astype("string")
    .str.replace(r"\\n|\n", "", regex=True)
    .str.strip()
)

print("Clean practice statuses:")
print(
    stage1_features["initial_practice_status_clean"]
    .value_counts(dropna=False)
)

print("\nExact representations:")
for value in stage1_features["initial_practice_status_clean"].dropna().unique():
    print(repr(value))

Clean practice statuses:
initial_practice_status_clean
Limited Participation in Practice    13
Did Not Participate In Practice       5
Full Participation in Practice        4
Out (Definitely Will Not Play)        1
                                      1
Name: count, dtype: Int64

Exact representations:
'Full Participation in Practice'
'Did Not Participate In Practice'
'Limited Participation in Practice'
'Out (Definitely Will Not Play)'
''


In [24]:
# Step 23 — rebuild practice severity indicator after cleaning

stage1_features["practice_dnp_or_out"] = (
    stage1_features["initial_practice_status_clean"]
    .isin([
        "Did Not Participate in Practice",
        "Out (Definitely Will Not Play)"
    ])
    .astype(int)
)

print("Practice DNP/Out vs target:")
print(
    pd.crosstab(
        stage1_features["practice_dnp_or_out"],
        stage1_features["missed_game_target"],
        margins=True
    )
)

print("\nFlagged episodes:")
print(
    stage1_features.loc[
        stage1_features["practice_dnp_or_out"] == 1,
        [
            "player",
            "season",
            "start_week",
            "initial_practice_status_clean",
            "initial_report_status",
            "missed_game_target"
        ]
    ].to_string(index=False)
)

Practice DNP/Out vs target:
missed_game_target   0   1  All
practice_dnp_or_out            
0                    9  14   23
1                    0   1    1
All                  9  15   24

Flagged episodes:
        player  season  start_week  initial_practice_status_clean initial_report_status  missed_game_target
Johnny Manziel    2014          17 Out (Definitely Will Not Play)                   Out                   1


In [25]:
# Step 24 — compare corrected onset severity indicators

print(
    pd.crosstab(
        stage1_features["practice_dnp_or_out"],
        stage1_features["report_out_or_doubtful"],
        margins=True
    )
)

print("\nOutcome rates:")

for feature in ["practice_dnp_or_out", "report_out_or_doubtful"]:
    summary = (
        stage1_features
        .groupby(feature)["missed_game_target"]
        .agg(["count", "sum", "mean"])
    )

    print(f"\n{feature}:")
    print(summary)

report_out_or_doubtful   0  1  All
practice_dnp_or_out               
0                       17  6   23
1                        0  1    1
All                     17  7   24

Outcome rates:

practice_dnp_or_out:
                     count  sum      mean
practice_dnp_or_out                      
0                       23   14  0.608696
1                        1    1  1.000000

report_out_or_doubtful:
                        count  sum      mean
report_out_or_doubtful                      
0                          17    8  0.470588
1                           7    7  1.000000


In [26]:
# Step 25 — robustly normalize practice-status text

stage1_features["initial_practice_status_clean"] = (
    stage1_features["initial_practice_status"]
    .astype("string")
    .str.replace(r"\\n|\n", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print("Status representations and lengths:\n")

for value in stage1_features["initial_practice_status_clean"].dropna().unique():
    print(
        repr(value),
        "| length:",
        len(value),
        "| codepoints:",
        [ord(c) for c in value]
    )

Status representations and lengths:

'Full Participation in Practice' | length: 30 | codepoints: [70, 117, 108, 108, 32, 80, 97, 114, 116, 105, 99, 105, 112, 97, 116, 105, 111, 110, 32, 105, 110, 32, 80, 114, 97, 99, 116, 105, 99, 101]
'Did Not Participate In Practice' | length: 31 | codepoints: [68, 105, 100, 32, 78, 111, 116, 32, 80, 97, 114, 116, 105, 99, 105, 112, 97, 116, 101, 32, 73, 110, 32, 80, 114, 97, 99, 116, 105, 99, 101]
'Limited Participation in Practice' | length: 33 | codepoints: [76, 105, 109, 105, 116, 101, 100, 32, 80, 97, 114, 116, 105, 99, 105, 112, 97, 116, 105, 111, 110, 32, 105, 110, 32, 80, 114, 97, 99, 116, 105, 99, 101]
'Out (Definitely Will Not Play)' | length: 30 | codepoints: [79, 117, 116, 32, 40, 68, 101, 102, 105, 110, 105, 116, 101, 108, 121, 32, 87, 105, 108, 108, 32, 78, 111, 116, 32, 80, 108, 97, 121, 41]
'' | length: 0 | codepoints: []


In [27]:
# Step 26 — robust practice severity indicator

practice_text = (
    stage1_features["initial_practice_status_clean"]
    .fillna("")
    .str.lower()
)

stage1_features["practice_dnp_or_out"] = (
    practice_text.str.contains("did not participate", regex=False)
    |
    practice_text.str.contains("definitely will not play", regex=False)
).astype(int)

print("Practice DNP/Out vs target:")
print(
    pd.crosstab(
        stage1_features["practice_dnp_or_out"],
        stage1_features["missed_game_target"],
        margins=True
    )
)

print("\nFlagged episodes:")
print(
    stage1_features.loc[
        stage1_features["practice_dnp_or_out"] == 1,
        [
            "player",
            "season",
            "start_week",
            "initial_practice_status_clean",
            "initial_report_status",
            "missed_game_target"
        ]
    ].to_string(index=False)
)

Practice DNP/Out vs target:
missed_game_target   0   1  All
practice_dnp_or_out            
0                    9   9   18
1                    0   6    6
All                  9  15   24

Flagged episodes:
        player  season  start_week   initial_practice_status_clean initial_report_status  missed_game_target
  Michael Vick    2015           7 Did Not Participate In Practice                   Out                   1
    Jay Cutler    2015           3 Did Not Participate In Practice                   Out                   1
    Brad Smith    2011          17 Did Not Participate In Practice                   Out                   1
Blaine Gabbert    2013           6 Did Not Participate In Practice                   Out                   1
Johnny Manziel    2014          17  Out (Definitely Will Not Play)                   Out                   1
  Dak Prescott    2024          10 Did Not Participate In Practice                   Out                   1


In [28]:
# Step 27 — compare corrected onset severity indicators

severity_compare = pd.crosstab(
    stage1_features["practice_dnp_or_out"],
    stage1_features["report_out_or_doubtful"],
    margins=True
)

print("Practice DNP/Out vs Report Out/Doubtful:")
print(severity_compare)

print("\nCases where indicators disagree:")

disagreement = stage1_features.loc[
    stage1_features["practice_dnp_or_out"]
    != stage1_features["report_out_or_doubtful"],
    [
        "player",
        "season",
        "start_week",
        "initial_practice_status_clean",
        "initial_report_status",
        "practice_dnp_or_out",
        "report_out_or_doubtful",
        "missed_game_target"
    ]
]

print(
    disagreement
    .sort_values(["season", "player"])
    .to_string(index=False)
)

Practice DNP/Out vs Report Out/Doubtful:
report_out_or_doubtful   0  1  All
practice_dnp_or_out               
0                       17  1   18
1                        0  6    6
All                     17  7   24

Cases where indicators disagree:
      player  season  start_week     initial_practice_status_clean initial_report_status  practice_dnp_or_out  report_out_or_doubtful  missed_game_target
Daniel Jones    2020          13 Limited Participation in Practice              Doubtful                    0                       1                   1


In [29]:
# Step 28 — compare candidate Stage 1 features

candidate_features = [
    "practice_dnp_or_out",
    "report_out_or_doubtful",
    "prior_hamstring_episodes",
    "pre3_pass_attempts_avg",
    "pre3_carries_avg",
    "pre3_total_actions_avg"
]

print("Candidate feature comparison:\n")

for feature in candidate_features:

    print("=" * 60)
    print(feature)

    if stage1_features[feature].nunique(dropna=True) <= 5:
        print(
            stage1_features
            .groupby(feature, dropna=False)["missed_game_target"]
            .agg(
                episodes="count",
                missed_games="sum",
                missed_game_rate="mean"
            )
        )

    else:
        print(
            stage1_features
            .groupby("missed_game_target")[feature]
            .agg(["count", "mean", "median", "std"])
            .round(2)
        )

    print()

Candidate feature comparison:

practice_dnp_or_out
                     episodes  missed_games  missed_game_rate
practice_dnp_or_out                                          
0                          18             9               0.5
1                           6             6               1.0

report_out_or_doubtful
                        episodes  missed_games  missed_game_rate
report_out_or_doubtful                                          
0                             17             8          0.470588
1                              7             7          1.000000

prior_hamstring_episodes
                          episodes  missed_games  missed_game_rate
prior_hamstring_episodes                                          
0                               19            12          0.631579
1                                4             2          0.500000
2                                1             1          1.000000

pre3_pass_attempts_avg
                    count   mean

In [30]:
# Step 29 — build final Stage 1 modeling feature table

stage1_model_data = stage1_features[
    [
        "player",
        "gsis_id",
        "season",
        "start_week",
        "team",

        # Primary onset-severity predictor
        "report_out_or_doubtful",

        # Historical context
        "prior_hamstring_episodes",

        # Pre-injury workload
        "pre3_total_actions_avg",
        "pre3_games_used",

        # Outcome
        "missed_game_target"
    ]
].copy()

print("Stage 1 modeling shape:", stage1_model_data.shape)

print("\nMissing values:")
print(stage1_model_data.isna().sum())

print("\nTarget distribution:")
print(
    stage1_model_data["missed_game_target"]
    .value_counts()
    .sort_index()
)

print("\nFinal modeling table:")
print(
    stage1_model_data
    .sort_values(["season", "player", "start_week"])
    .to_string(index=False)
)

Stage 1 modeling shape: (24, 10)

Missing values:
player                      0
gsis_id                     0
season                      0
start_week                  0
team                        0
report_out_or_doubtful      0
prior_hamstring_episodes    0
pre3_total_actions_avg      1
pre3_games_used             0
missed_game_target          0
dtype: int64

Target distribution:
missed_game_target
0     9
1    15
Name: count, dtype: int64

Final modeling table:
         player    gsis_id  season  start_week team  report_out_or_doubtful  prior_hamstring_episodes  pre3_total_actions_avg  pre3_games_used  missed_game_target
    Vince Young 00-0024218    2009          15  TEN                       0                         0               34.333333                3                   0
  David Garrard 00-0021231    2010          15  JAX                       0                         0               32.333333                3                   0
       Joe Webb 00-0027796    2010        

In [31]:
# Step 30 — inspect final Stage 1 predictors

final_numeric_features = [
    "report_out_or_doubtful",
    "prior_hamstring_episodes",
    "pre3_total_actions_avg",
    "pre3_games_used",
    "missed_game_target"
]

print("Correlation matrix:")
print(
    stage1_model_data[final_numeric_features]
    .corr()
    .round(2)
)

print("\nFinal feature summary:")
print(
    stage1_model_data[
        [
            "report_out_or_doubtful",
            "prior_hamstring_episodes",
            "pre3_total_actions_avg",
            "pre3_games_used"
        ]
    ]
    .describe()
    .round(2)
)

Correlation matrix:
                          report_out_or_doubtful  prior_hamstring_episodes  \
report_out_or_doubtful                      1.00                     -0.13   
prior_hamstring_episodes                   -0.13                      1.00   
pre3_total_actions_avg                     -0.48                      0.09   
pre3_games_used                             0.07                      0.03   
missed_game_target                          0.50                      0.04   

                          pre3_total_actions_avg  pre3_games_used  \
report_out_or_doubtful                     -0.48             0.07   
prior_hamstring_episodes                    0.09             0.03   
pre3_total_actions_avg                      1.00             0.35   
pre3_games_used                             0.35             1.00   
missed_game_target                         -0.44             0.15   

                          missed_game_target  
report_out_or_doubtful                  0.50  
pr

In [32]:
# Step 31 — flag missing pre-injury workload

stage1_model_data["pre3_workload_missing"] = (
    stage1_model_data["pre3_total_actions_avg"]
    .isna()
    .astype(int)
)

print("Workload missing indicator:")
print(
    stage1_model_data["pre3_workload_missing"]
    .value_counts()
    .sort_index()
)

print("\nMissing workload episode:")
print(
    stage1_model_data.loc[
        stage1_model_data["pre3_workload_missing"] == 1,
        [
            "player",
            "season",
            "start_week",
            "pre3_games_used",
            "pre3_total_actions_avg",
            "missed_game_target"
        ]
    ].to_string(index=False)
)

Workload missing indicator:
pre3_workload_missing
0    23
1     1
Name: count, dtype: int64

Missing workload episode:
  player  season  start_week  pre3_games_used  pre3_total_actions_avg  missed_game_target
Joe Webb    2010          14                0                     NaN                   0


In [33]:
# Step 32 — save final Stage 1 engineered dataset

from pathlib import Path

DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

stage1_path = DATA_DIR / "stage1_engineered_features.csv"

stage1_model_data.to_csv(
    stage1_path,
    index=False
)

print("Saved to:", stage1_path)
print("Rows:", len(stage1_model_data))
print("Columns:", len(stage1_model_data.columns))

Saved to: ../data/stage1_engineered_features.csv
Rows: 24
Columns: 11


In [34]:
# Step 33 — final Stage 1 integrity checks

assert len(stage1_model_data) == 24
assert stage1_model_data["missed_game_target"].isna().sum() == 0
assert stage1_model_data["missed_game_target"].sum() == 15

assert (
    stage1_model_data[
        ["player", "season", "start_week"]
    ]
    .duplicated()
    .sum()
    == 0
)

assert stage1_model_data["pre3_workload_missing"].sum() == 1

assert (
    stage1_model_data["report_out_or_doubtful"]
    .isin([0, 1])
    .all()
)

print("✓ Stage 1 dataset passed all integrity checks.")
print("✓ 24 unique QB hamstring episodes.")
print("✓ 15 missed-game / 9 no-missed-game outcomes.")
print("✓ Feature engineering complete.")

✓ Stage 1 dataset passed all integrity checks.
✓ 24 unique QB hamstring episodes.
✓ 15 missed-game / 9 no-missed-game outcomes.
✓ Feature engineering complete.
